In [0]:
# ============================================================
# NOTEBOOK 01 — BRONZE: PREPARATION AND FEATURE GROUPING
# Vermont School - Early Warning System V2
#
# INPUT:  bronze/anon/24_25/ and bronze/anon/25_26/
# OUTPUT: bronze/prepared/24_25/ and bronze/prepared/25_26/
#         Unified subject groups: Science, I&S, etc.
# ============================================================

%pip install lxml

In [0]:
import pandas as pd
import numpy as np
import os

BRONZE  = "/Volumes/workspace/vermont/bronze"
ANON_24 = f"{BRONZE}/anon/24_25"
ANON_25 = f"{BRONZE}/anon/25_26"
PREP_24 = f"{BRONZE}/prepared/24_25"
PREP_25 = f"{BRONZE}/prepared/25_26"

for carpeta in [PREP_24, PREP_25]:
    dbutils.fs.mkdirs(carpeta)

# ── SUBJECT GROUPING MAP ──
# To add a new subject next year, just add it to the list here
GROUPS = {
    'Science': [
        'Integrated Science', 'Life Science', 'Physical Science',
        'Biology', 'Chemistry', 'Physics'
    ],
    'I_and_S': [
        'Individuals and societies', 'Ciencias Políticas'
    ],
    'Mathematics':          ['Mathematics'],
    'English':              ['English'],
    'Lengua_Castellana':    ['Lengua Castellana'],
    'Mandarin':             ['Mandarín'],
    'Financial_Maths':      ['Financial Maths'],
    'ICT_STEM':             ['ICT/STEM'],
    'Physical_Education':   ['Educación Física'],
    'Research_Methodology': ['Research methodology'],
}

EXCLUDE = [
    'Promedio', 'Puesto',
    'Vermont Skills - Arts',
    'Vermont Skills - STEM',
    'Vermont Skills - Sports'
]

WEIGHTS = {
    'Primer Trimestre':  0.30,
    'Segundo Trimestre': 0.30,
    'Tercer Trimestre':  0.40
}

print("✓ Configuration loaded")
print(f"  Subject groups: {len(GROUPS)}")
for group, subjects in GROUPS.items():
    print(f"    {group}: {subjects}")

In [0]:
# CELDA 3 — Subject grouping and grade calculation function

def process_grades(anon_path, year_label):
    """
    Reads anonymized grade files, groups subjects and
    calculates weighted cumulative grades per student.
    Returns a unified DataFrame with one row per student.
    """
    cursos = ["7A","7B","8A","8B","9A","9B"]
    records = []

    for curso in cursos:
        filepath = f"{anon_path}/notas_{curso}.csv"
        try:
            df = pd.read_csv(filepath)
        except Exception as e:
            print(f"  Warning: {curso} not found — {e}")
            continue

        col_name   = df.columns[0]
        col_period = df.columns[1]

        # Filter only the three terms
        df_terms = df[df[col_period].isin(WEIGHTS.keys())].copy()

        for student in df_terms[col_name].dropna().unique():
            df_stu = df_terms[df_terms[col_name] == student]
            rec = {
                'student_id':  str(student).strip(),
                'section_anon': curso,
                'grade':        curso[0],  # '7', '8', '9'
                'year':         year_label
            }

            # Calculate grouped subject grades per term and cumulative
            for group_name, subject_list in GROUPS.items():
                # Available subjects for this student/section
                avail = [s for s in subject_list if s in df_stu.columns]

                if not avail:
                    # No subjects available for this group/grade
                    for t in [1, 2, 3]:
                        rec[f'{group_name}_T{t}'] = np.nan
                    rec[f'{group_name}_cum'] = np.nan
                    continue

                # Per term: average of available subjects
                for period, peso in WEIGHTS.items():
                    t_num = list(WEIGHTS.keys()).index(period) + 1
                    row = df_stu[df_stu[col_period] == period]
                    if row.empty:
                        rec[f'{group_name}_T{t_num}'] = np.nan
                        continue
                    vals = []
                    for s in avail:
                        try:
                            v = float(row[s].values[0])
                            vals.append(v)
                        except:
                            pass
                    rec[f'{group_name}_T{t_num}'] = round(np.mean(vals), 4) if vals else np.nan

                # Cumulative weighted grade (30/30/40)
                cum = 0
                total_w = 0
                for period, peso in WEIGHTS.items():
                    t_num = list(WEIGHTS.keys()).index(period) + 1
                    v = rec.get(f'{group_name}_T{t_num}')
                    if v is not None and not np.isnan(v):
                        cum += v * peso
                        total_w += peso
                rec[f'{group_name}_cum'] = round(cum / total_w, 4) if total_w > 0 else np.nan

            # Risk level based on cumulative grades
            cum_grades = [rec[f'{g}_cum'] for g in GROUPS if not np.isnan(rec.get(f'{g}_cum', np.nan))]
            below_4 = sum(1 for v in cum_grades if v < 4.0)

            rec['n_subjects_below_4'] = below_4
            rec['avg_cumulative']     = round(np.nanmean(cum_grades), 4) if cum_grades else np.nan
            rec['min_cumulative']     = round(min(cum_grades), 4) if cum_grades else np.nan
            rec['risk_level'] = (
                'critical'     if below_4 >= 3 else
                'recovery'     if below_4 >= 1 else
                'no_risk'
            )

            records.append(rec)

    df_out = pd.DataFrame(records)
    print(f"  {year_label}: {len(df_out)} students, {len(df_out.columns)} columns")
    return df_out

print("✓ Function defined")

In [0]:
# CELDA 4 — Process both years and save to bronze/prepared/

print("Processing grades...")
df_grades_24 = process_grades(ANON_24, '24_25')
df_grades_25 = process_grades(ANON_25, '25_26')

print(f"\nRisk distribution 24-25:")
print(df_grades_24['risk_level'].value_counts().to_string())
print(f"\nRisk distribution 25-26:")
print(df_grades_25['risk_level'].value_counts().to_string())

# ── Process attendance ──
print("\nProcessing attendance...")

ATTEND_COLS = ['Estudiante', 'Ausencia Clase', 'Ausencia Clase con Excusa',
               'Ausencia Colegio', 'Ausencia Colegio con Excusa',
               'Retardo', 'Salida Temprano', 'Total']

da24 = pd.read_csv(f"{ANON_24}/asistencia.csv")
da25 = pd.read_csv(f"{ANON_25}/asistencia.csv")

da24 = da24[[c for c in ATTEND_COLS if c in da24.columns]].rename(columns={
    'Estudiante':                   'student_id',
    'Ausencia Clase':               'absence_class',
    'Ausencia Clase con Excusa':    'absence_class_excused',
    'Ausencia Colegio':             'absence_school',
    'Ausencia Colegio con Excusa':  'absence_school_excused',
    'Retardo':                      'late',
    'Salida Temprano':              'early_leave',
    'Total':                        'total_absences'
})
da25 = da25[[c for c in ATTEND_COLS if c in da25.columns]].rename(columns={
    'Estudiante':                   'student_id',
    'Ausencia Clase':               'absence_class',
    'Ausencia Clase con Excusa':    'absence_class_excused',
    'Ausencia Colegio':             'absence_school',
    'Ausencia Colegio con Excusa':  'absence_school_excused',
    'Retardo':                      'late',
    'Salida Temprano':              'early_leave',
    'Total':                        'total_absences'
})
da24['student_id'] = da24['student_id'].astype(str).str.strip()
da25['student_id'] = da25['student_id'].astype(str).str.strip()
print(f"  24-25: {len(da24)} students")
print(f"  25-26: {len(da25)} students")

# ── Process disciplinary records ──
print("\nProcessing disciplinary records...")
import re

def extract_fault_number(text):
    if pd.isna(text): return None
    m = re.match(r'^\s*(\d+)\.', str(text).strip())
    return int(m.group(1)) if m else None

def process_disciplinary(anon_path, year_label):
    if year_label == '24_25':
        df = pd.read_csv(f"{anon_path}/convivencia.csv")
        # Identify type from column
        col_tipo = [c for c in df.columns if 'Tipo de Falta' in c or 'Tipo' in c]
        col_tipo = col_tipo[1] if len(col_tipo) > 1 else col_tipo[0]
        df_f1 = df[df[col_tipo].str.contains('Tipo I', na=False) & 
                   ~df[col_tipo].str.contains('Tipo II', na=False)]
        df_f2 = df[df[col_tipo].str.contains('Tipo II', na=False)]
    else:
        df_f1 = pd.read_csv(f"{anon_path}/convivencia_tipo1.csv")
        df_f2 = pd.read_csv(f"{anon_path}/convivencia_tipo2.csv")

    # Extract fault number
    col_falta = [c for c in df_f1.columns if 'Faltas' in c or 'Situacion' in c or 'Falta o' in c]
    col_falta = col_falta[0] if col_falta else None

    if col_falta:
        df_f1['fault_number'] = df_f1[col_falta].apply(extract_fault_number)
    else:
        df_f1['fault_number'] = None

    # Aggregate per student
    f1_agg = df_f1.groupby('Código').agg(
        n_f1=('Id','count'),
        n_fault_types=('fault_number','nunique'),
        top_fault=('fault_number', lambda x: x.mode()[0] if x.notna().any() else None)
    ).reset_index().rename(columns={'Código':'student_id'})

    f2_agg = df_f2.groupby('Código').agg(
        n_f2=('Id','count')
    ).reset_index().rename(columns={'Código':'student_id'})

    f1_agg['student_id'] = f1_agg['student_id'].astype(str).str.strip()
    f2_agg['student_id'] = f2_agg['student_id'].astype(str).str.strip()

    return f1_agg, f2_agg

f1_24, f2_24 = process_disciplinary(ANON_24, '24_25')
f1_25, f2_25 = process_disciplinary(ANON_25, '25_26')

print(f"  24-25 — F1: {len(f1_24)} students, F2: {len(f2_24)} students")
print(f"  25-26 — F1: {len(f1_25)} students, F2: {len(f2_25)} students")

# ── Merge all sources ──
print("\nMerging all sources...")

def merge_year(df_grades, da, f1, f2):
    df = df_grades.merge(da, on='student_id', how='left')
    df = df.merge(f1, on='student_id', how='left')
    df = df.merge(f2, on='student_id', how='left')
    df['n_f1'] = df['n_f1'].fillna(0).astype(int)
    df['n_f2'] = df['n_f2'].fillna(0).astype(int)
    df['n_fault_types'] = df['n_fault_types'].fillna(0).astype(int)
    return df

df_24 = merge_year(df_grades_24, da24, f1_24, f2_24)
df_25 = merge_year(df_grades_25, da25, f1_25, f2_25)

# ── Save as Parquet ──
print("\nSaving to bronze/prepared/...")
spark.createDataFrame(df_24).write.mode("overwrite").parquet(PREP_24)
spark.createDataFrame(df_25).write.mode("overwrite").parquet(PREP_25)

print(f"\n{'='*55}")
print(f"BRONZE PREPARATION COMPLETE")
print(f"{'='*55}")
print(f"  24-25: {len(df_24)} students × {len(df_24.columns)} cols → {PREP_24}")
print(f"  25-26: {len(df_25)} students × {len(df_25.columns)} cols → {PREP_25}")
print(f"\n  Risk distribution 24-25:")
print(df_24['risk_level'].value_counts().to_string())
print(f"\n  Risk distribution 25-26:")
print(df_25['risk_level'].value_counts().to_string())